# Experiment 026 — 30M Cell Granularity Differentiation

Formal fixed-capacity G={1,2,4,8} ablation for Kaggle **Save Version → Run All**. Target hardware is Tesla T4×2. The job has an 8-hour global wall budget, a 30-minute reporting/publication reserve, automatic per-arm checkpoint/resume, and a balanced four-domain 20M-token continuation. Persistent growth is disabled.


In [ ]:
import subprocess, sys
from pathlib import Path

ROOT = Path('/kaggle/working/mini-cells')
BRANCH = 'codex/experiment-026-cell-granularity'
REPO = 'https://github.com/ArcheLabs/mini-cells.git'

if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, str(ROOT)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=ROOT, check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], cwd=ROOT, check=True)
HEAD = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip()
TREE = subprocess.check_output(['git', 'rev-parse', 'HEAD^{tree}'], cwd=ROOT, text=True).strip()
DIRTY = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=ROOT, text=True).strip()
print({'HEAD': HEAD, 'tree': TREE, 'tracked_dirty': bool(DIRTY)})
assert not DIRTY


In [ ]:
subprocess.run([
    sys.executable, '-m', 'pytest',
    'tests/research/02-self-organization/test_developmental_tissue.py',
    'tests/research/03-routing-and-growth/test_experiment_026_cell_granularity.py',
    '-q',
], cwd=ROOT, check=True)
subprocess.run([sys.executable, 'scripts/research/run_experiment_026_cell_granularity_smoke.py'], cwd=ROOT, check=True)


In [ ]:
import torch
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print({'gpu_count': torch.cuda.device_count(), 'gpus': gpu_names})
assert torch.cuda.device_count() >= 2, 'Formal Experiment 026 expects T4×2'


## One-shot formal run

All four arms start from the same retained 30M source and fixed 12-root CLM. Two arms run concurrently. Incomplete arms rotate through 2.25-hour worker slices and resume automatically while the global budget remains.


In [ ]:
subprocess.run([
    sys.executable,
    'scripts/research/run_experiment_026_cell_granularity.py',
    '--total-wall-hours', '8',
    '--finalization-reserve-minutes', '30',
    '--worker-slice-hours', '2.25',
], cwd=ROOT, check=True)


In [ ]:
import json
from IPython.display import Image, display

OUT = ROOT / 'results' / 'experiment-026-cell-granularity'
summary_path = OUT / 'worker-summary.json'
summary = json.loads(summary_path.read_text()) if summary_path.is_file() else {}
print(json.dumps(summary, indent=2))
decision_path = OUT / 'decision.json'
if decision_path.is_file():
    decision = json.loads(decision_path.read_text())
    print(json.dumps(decision, indent=2))
    for name in ['performance-by-granularity.png', 'differentiation-by-granularity.png', 'granularity-frontier.png']:
        path = OUT / name
        if path.is_file():
            display(Image(filename=str(path)))
else:
    print('Formal run ended partial inside the 8h budget; checkpoints and partial tables were preserved.')


## Automatic publication

Publication runs only when all four granularity arms completed and `decision.json` exists. The frozen `protocol.json` is copied into the result bundle before training, so publication does not depend on reconstructing provenance afterward. A publication failure does not erase completed Kaggle outputs.


In [ ]:
if bool(summary.get('complete')) and decision_path.is_file():
    publish = subprocess.run([
        sys.executable,
        'scripts/research/publish_experiment_026_cell_granularity.py',
        '--push',
    ], cwd=ROOT, text=True, capture_output=True)
    print(publish.stdout)
    if publish.returncode != 0:
        print('Publication failed; completed Kaggle outputs remain available.')
        print(publish.stderr)
else:
    print('Publication skipped because the formal run is incomplete.')
